# SQL joins and grouped revenue

демо-реконструкция подхода на синтетических данных

Use RETAIL_DATASET for the public UCI source; otherwise the run explicitly uses fictional data. See DATA_SOURCES.md.


In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from src.data import load_transactions
transactions, source_label = load_transactions(os.getenv('RETAIL_DATASET'))
print(source_label)
print('Rows:', len(transactions))

Synthetic fallback fixture, not open-data results
Rows: 134


In [2]:
import sqlite3
conn = sqlite3.connect(':memory:')
transactions.to_sql('transactions', conn, index=False)
products = transactions[['product']].drop_duplicates().copy()
products['category'] = ['Category '+str(i%3) for i in range(len(products))]
products.to_sql('products', conn, index=False)
query = '''SELECT p.category, COUNT(*) AS lines, SUM(t.revenue) AS revenue
FROM transactions t JOIN products p ON t.product=p.product
GROUP BY p.category ORDER BY revenue DESC'''
result = __import__('pandas').read_sql_query(query, conn)
assert abs(result.revenue.sum()-transactions.revenue.sum()) < 0.01
print(result.to_string(index=False))
conn.close()

  category  lines  revenue
Category 1     54     3951
Category 0     53     3522
Category 2     27     1962
